# Research: Is Variant D Orbit-Aware SG Regularization Safer?

This notebook tests the main pre-training gate for Variant D:

```text
proj_gt_direct = ||k(L) - Pi_G(k(L))||
proj_gt_orbit  = min_U ||k(U L) - Pi_G(k(U L))||
```

If `proj_gt_orbit << proj_gt_direct`, then the data problem is partly an arbitrary primitive-basis chart problem, and an orbit-aware SG auxiliary is safer than direct projection.

This notebook does **not** train a model and does **not** change dataset targets. It only measures whether the regularizer is mathematically promising on ground-truth lattices.


## What This Tests

Variant D keeps the raw paired training data:

```text
(F_raw, k_raw, A, G)
```

and regularizes the lattice equivalence class:

```text
[L] = {U L : U in GL(3, Z), det(U)=+/-1}
```

The notebook checks three things:

- Whether `proj_gt_orbit` is much smaller than `proj_gt_direct`.
- Whether the best `U*` basis transform preserves Cartesian positions with `F* = F inv(U*) mod 1`.
- How dangerous direct projection `k -> Pi_G(k)` would be if `F` were left unchanged.


In [1]:
from __future__ import annotations

import csv
import itertools
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "kldmPlus").exists():
    raise RuntimeError(f"Could not locate repo root from cwd={Path.cwd()}")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.symmetry.latticeSymmetry import LatticeSymmetry

CONFIG_PATH = ROOT / "configs" / "kldm_plus" / "mp_20" / "mp20_diffcsp_k_x0_soft_lattice_laptop.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen


{'experiment_name': 'plus_mp20_diffcsp_k_x0_soft_lattice_laptop',
 'sampler_config': '../../samplers/predictor_corrector.yaml',
 'dataset': {'task': 'csp',
  'name': 'mp20',
  'lattice_representation': 'diffcsp_k',
  'root': None,
  'batch_size': 32,
  'num_workers': 0,
  'pin_memory': False,
  'train_subset_fraction': 0.1,
  'train_subset_seed': 2002,
  'train_subset_strategy': 'balanced_space_group'},
 'time_sampler': {'type': 'uniform'},
 'model': {'eps': 1e-06,
  'lambda_l': 1.0,
  'lattice_parameterization': 'x0',
  'lattice_diffusion_type': 'VP',
  'lattice_representation': 'diffcsp_k',
  'lattice_sg_lambda': 1,
  'lattice_sg_normalize': True,
  'lattice_sg_time_weight': 'alpha_squared',
  'wrapped_normal_K': 3,
  'tdm_n_sigmas': 512,
  'tdm_compute_sigma_norm': True,
  'tdm_velocity_scale': 0.15915494309189535,
  'tdm_sigma_norm_estimator': 'quadrature',
  'tdm_sigma_norm_density_K': 20,
  'tdm_sigma_norm_grid_points': 1025,
  'tdm_sigma_norm_mc_samples': 2000,
  'score_network'

In [2]:
DEVICE = torch.device("cpu")
torch.set_grad_enabled(False)

dataset_cfg = cfg["dataset"]
model_cfg = cfg["model"]

task = CSPTask(
    dataset_name=dataset_cfg["name"],
    lattice_parameterization=model_cfg["lattice_parameterization"],
    lattice_representation=dataset_cfg["lattice_representation"],
)
root = resolve_data_root(dataset_cfg.get("root"))
sym = LatticeSymmetry(eps=float(model_cfg.get("eps", 1e-8))).to(DEVICE)

print("repo", ROOT)
print("data root", root)
print("config", CONFIG_PATH)
print("lattice_representation", dataset_cfg["lattice_representation"])
print("lattice_sg_lambda", model_cfg.get("lattice_sg_lambda"))
print("lattice_sg_time_weight", model_cfg.get("lattice_sg_time_weight"))


repo /workspace
data root /workspace/data
config /workspace/configs/kldm_plus/mp_20/mp20_diffcsp_k_x0_soft_lattice_laptop.yaml
lattice_representation diffcsp_k
lattice_sg_lambda 1
lattice_sg_time_weight alpha_squared


## Helpers


In [3]:
def sg_family(sg: int) -> str:
    sg = int(sg)
    if 195 <= sg <= 230:
        return "cubic"
    if 143 <= sg <= 194:
        return "hex_trig"
    if 75 <= sg <= 142:
        return "tetragonal"
    if 16 <= sg <= 74:
        return "orthorhombic"
    if 3 <= sg <= 15:
        return "monoclinic"
    return "triclinic"


def get_structure_id(dataset, i: int) -> str | None:
    try:
        return str(dataset.data.structure_id[i])
    except Exception:
        return None


def load_raw_sg_map(split: str) -> dict[str, int]:
    path = root / "mp_20" / "raw" / f"{split}.csv"
    if not path.exists():
        return {}
    mapping = {}
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            value = row.get("spacegroup.number")
            if value not in {None, ""}:
                mapping[str(row["material_id"])] = int(float(value))
    return mapping


def lengths_angles_volume(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cell = cell.reshape(3, 3)
    lengths = torch.linalg.norm(cell, dim=1)
    alpha = torch.acos(torch.clamp(torch.dot(cell[1], cell[2]) / (lengths[1] * lengths[2]).clamp_min(1e-12), -1.0, 1.0))
    beta = torch.acos(torch.clamp(torch.dot(cell[0], cell[2]) / (lengths[0] * lengths[2]).clamp_min(1e-12), -1.0, 1.0))
    gamma = torch.acos(torch.clamp(torch.dot(cell[0], cell[1]) / (lengths[0] * lengths[1]).clamp_min(1e-12), -1.0, 1.0))
    angles_deg = torch.rad2deg(torch.stack([alpha, beta, gamma]))
    volume = torch.det(cell).abs().clamp_min(1e-12)
    return lengths, angles_deg, volume


def k_from_lattice_matrix(cell: torch.Tensor) -> torch.Tensor:
    return sym.m2v(sym.de_so3(cell.reshape(-1, 3, 3)))


def direct_sg_errors(k: torch.Tensor, sg: torch.Tensor) -> dict[str, torch.Tensor]:
    proj = sym.proj_k_to_spacegroup(k, sg.reshape(-1).long())
    diff = k - proj
    return {
        "proj": proj,
        "abs_mean_all6": diff.abs().mean(dim=1),
        "mse_all6": diff.square().mean(dim=1),
        "l2": torch.linalg.vector_norm(diff, dim=1),
    }


## Build Unimodular Candidate Set

The default finite orbit uses entries in `{-1, 0, 1}` with determinant `+/-1`. This gives 6960 candidates and is enough for a first gate.


In [4]:
def generate_unimodular_candidates(values=(-1, 0, 1)) -> torch.Tensor:
    candidates = []
    eye = torch.eye(3, dtype=torch.float32)
    for vals in itertools.product(values, repeat=9):
        U = torch.tensor(vals, dtype=torch.float32).reshape(3, 3)
        det = int(round(torch.linalg.det(U).item()))
        if abs(det) == 1:
            candidates.append(U)
    candidates.sort(key=lambda U: 0 if torch.equal(U, eye) else 1)
    return torch.stack(candidates, dim=0)


U_CANDIDATES = generate_unimodular_candidates()
print("num_U_candidates", int(U_CANDIDATES.shape[0]))
print("identity_first", U_CANDIDATES[0].int().tolist())


num_U_candidates 6960
identity_first [[1, 0, 0], [0, 1, 0], [0, 0, 1]]


## Orbit-Aware SG Residual

For each `k`, decode to `L(k)`, try `L_U = U L(k)`, encode back to `k_U`, and compute the best SG residual.


In [5]:
def orbit_sg_errors(
    k: torch.Tensor,
    sg: torch.Tensor,
    candidates: torch.Tensor,
    *,
    candidate_chunk_size: int = 512,
) -> dict[str, torch.Tensor]:
    k = k.reshape(-1, 6).to(device=DEVICE, dtype=torch.float32)
    sg = sg.reshape(-1).to(device=DEVICE, dtype=torch.long)
    cell = sym.v2m(k).reshape(-1, 3, 3)
    batch_size = int(k.shape[0])

    best_mse = torch.full((batch_size,), float("inf"), device=DEVICE)
    best_abs = torch.full((batch_size,), float("inf"), device=DEVICE)
    best_l2 = torch.full((batch_size,), float("inf"), device=DEVICE)
    best_idx = torch.full((batch_size,), -1, device=DEVICE, dtype=torch.long)
    best_k = torch.empty_like(k)

    candidates = candidates.to(device=DEVICE, dtype=cell.dtype)
    batch_indices = torch.arange(batch_size, device=DEVICE)
    for start in range(0, int(candidates.shape[0]), int(candidate_chunk_size)):
        U = candidates[start:start + int(candidate_chunk_size)]
        # Shape: [M, B, 3, 3], row-vector convention L_U = U L.
        cell_u = torch.einsum("mij,bjk->mbik", U, cell)
        k_u = k_from_lattice_matrix(cell_u.reshape(-1, 3, 3)).reshape(U.shape[0], batch_size, 6)
        sg_rep = sg.unsqueeze(0).expand(U.shape[0], batch_size).reshape(-1)
        proj_u = sym.proj_k_to_spacegroup(k_u.reshape(-1, 6), sg_rep).reshape(U.shape[0], batch_size, 6)
        diff_u = k_u - proj_u
        mse_u = diff_u.square().mean(dim=2)
        abs_u = diff_u.abs().mean(dim=2)
        l2_u = torch.linalg.vector_norm(diff_u, dim=2)
        chunk_best_mse, chunk_arg = mse_u.min(dim=0)
        improve = chunk_best_mse < best_mse
        if improve.any():
            improved_batches = batch_indices[improve]
            src = chunk_arg[improve]
            best_mse[improve] = chunk_best_mse[improve]
            best_abs[improve] = abs_u[src, improved_batches]
            best_l2[improve] = l2_u[src, improved_batches]
            best_idx[improve] = start + src
            best_k[improve] = k_u[src, improved_batches]

    return {
        "best_mse_all6": best_mse,
        "best_abs_mean_all6": best_abs,
        "best_l2": best_l2,
        "best_idx": best_idx,
        "best_k": best_k,
    }


## Collect Dataset Samples


In [6]:
MAX_PER_SPLIT = 512
SPLITS = ["train", "val", "test"]

samples_by_key: dict[tuple[str, int], object] = {}
rows: list[dict] = []

for split in SPLITS:
    print(f"loading split={split}")
    dataset = task.fit_dataset(root=root, split=split, download=False)
    n = min(len(dataset), MAX_PER_SPLIT)
    raw_sg_map = load_raw_sg_map(split)
    for i in range(n):
        sample = dataset[i]
        samples_by_key[(split, i)] = sample
        k0 = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
        sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
        direct = direct_sg_errors(k0, sg)
        sid = get_structure_id(dataset, i)
        rows.append({
            "split": split,
            "idx": i,
            "structure_id": sid,
            "sg": int(sg.item()),
            "raw_csv_sg": raw_sg_map.get(sid),
            "family": sg_family(int(sg.item())),
            "proj_gt_direct_abs": float(direct["abs_mean_all6"].item()),
            "proj_gt_direct_mse": float(direct["mse_all6"].item()),
            "proj_gt_direct_l2": float(direct["l2"].item()),
        })

base_df = pd.DataFrame(rows)
display(base_df.head())
display(base_df.groupby(["split", "family"]).agg(
    n=("proj_gt_direct_abs", "size"),
    mean_direct_abs=("proj_gt_direct_abs", "mean"),
    p95_direct_abs=("proj_gt_direct_abs", lambda s: s.quantile(0.95)),
).sort_values(["split", "mean_direct_abs"], ascending=[True, False]))


loading split=train
dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
loading split=val
dataset_cache action=load dataset=mp_20 split=val reason=fresh path=/workspace/data/mp_20/processed/val
dataset_cache action=from_cache_path:start dataset=mp_20 split=val
dataset_cache action=from_cache_path:done dataset=mp_20 split=val
dataset_cache action=builder_build:start dataset=mp_20 split=val
dataset_cache action=builder_build:done dataset=mp_20 split=val
loading split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from

,split,idx,structure_id,sg,raw_csv_sg,family,proj_gt_direct_abs,proj_gt_direct_mse,proj_gt_direct_l2
0,train,0,mp-1221227,8,8,monoclinic,0.040951,0.007979,0.218798
1,train,1,mp-974729,139,139,tetragonal,0.139728,0.034203,0.453009
2,train,2,mp-1185360,225,225,cubic,0.163376,0.053384,0.565952
3,train,3,mp-1188861,62,62,orthorhombic,0.000000,0.000000,0.000000
4,train,4,mp-677272,122,122,tetragonal,0.163632,0.053391,0.565989


n  mean_direct_abs  p95_direct_abs
split family                                            
test  cubic         115         0.134963        0.163377
      hex_trig      109         0.114957        0.223981
      tetragonal    100         0.059766        0.147623
      monoclinic     74         0.045379        0.091789
      orthorhombic   93         0.040267        0.159163
      triclinic      21         0.000000        0.000000
train cubic         121         0.136372        0.163377
      hex_trig      116         0.110992        0.226678
      tetragonal     78         0.080701        0.163971
      orthorhombic   93         0.052359        0.162230
      monoclinic     85         0.043957        0.097963
      triclinic      19         0.000000        0.000000
val   cubic         110         0.124760        0.163377
      hex_trig      105         0.114004        0.223811
      tetragonal     77         0.072646        0.164287
      orthorhombic  120         0.046675        0.160505
      monoclinic     77         0.044651        0.096677
      triclinic      23         0.000000        0.000000

## Main Gate: `proj_gt_direct` vs `proj_gt_orbit`

Start with the worst direct-residual samples per split for speed. Set `ORBIT_MAX_PER_SPLIT = None` to scan all loaded samples.


In [7]:
ORBIT_MAX_PER_SPLIT = 128  # Increase to None for all loaded samples.
ORBIT_CANDIDATE_CHUNK_SIZE = 512

orbit_rows: list[dict] = []
for split in SPLITS:
    split_df = base_df[base_df["split"] == split].sort_values("proj_gt_direct_abs", ascending=False)
    if ORBIT_MAX_PER_SPLIT is not None:
        split_df = split_df.head(int(ORBIT_MAX_PER_SPLIT))
    print(f"orbit_scan split={split} n={len(split_df)} candidates={len(U_CANDIDATES)}")
    for _, row in split_df.iterrows():
        sample = samples_by_key[(str(row["split"]), int(row["idx"]))]
        k0 = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
        sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
        direct = direct_sg_errors(k0, sg)
        orbit = orbit_sg_errors(
            k0,
            sg,
            U_CANDIDATES,
            candidate_chunk_size=ORBIT_CANDIDATE_CHUNK_SIZE,
        )
        best_idx = int(orbit["best_idx"].item())
        best_U = U_CANDIDATES[best_idx]
        orbit_rows.append({
            **row.to_dict(),
            "proj_gt_direct_abs": float(direct["abs_mean_all6"].item()),
            "proj_gt_direct_mse": float(direct["mse_all6"].item()),
            "proj_gt_direct_l2": float(direct["l2"].item()),
            "proj_gt_orbit_abs": float(orbit["best_abs_mean_all6"].item()),
            "proj_gt_orbit_mse": float(orbit["best_mse_all6"].item()),
            "proj_gt_orbit_l2": float(orbit["best_l2"].item()),
            "orbit_abs_improvement": float(direct["abs_mean_all6"].item() - orbit["best_abs_mean_all6"].item()),
            "orbit_mse_improvement": float(direct["mse_all6"].item() - orbit["best_mse_all6"].item()),
            "orbit_relative_abs": float(orbit["best_abs_mean_all6"].item() / max(direct["abs_mean_all6"].item(), 1e-12)),
            "best_U_index_gt": best_idx,
            "best_U_det_gt": int(round(torch.linalg.det(best_U).item())),
            "best_U_gt": best_U.round().int().tolist(),
        })

orbit_df = pd.DataFrame(orbit_rows)
display(orbit_df.head())
display(orbit_df.groupby(["split", "family"]).agg(
    n=("proj_gt_direct_abs", "size"),
    mean_direct_abs=("proj_gt_direct_abs", "mean"),
    mean_orbit_abs=("proj_gt_orbit_abs", "mean"),
    median_orbit_relative_abs=("orbit_relative_abs", "median"),
    mean_abs_improvement=("orbit_abs_improvement", "mean"),
    p95_orbit_abs=("proj_gt_orbit_abs", lambda s: s.quantile(0.95)),
).sort_values(["split", "mean_direct_abs"], ascending=[True, False]))


orbit_scan split=train n=128 candidates=6960
orbit_scan split=val n=128 candidates=6960
orbit_scan split=test n=128 candidates=6960


,split,idx,structure_id,sg,raw_csv_sg,family,proj_gt_direct_abs,proj_gt_direct_mse,proj_gt_direct_l2,proj_gt_orbit_abs,proj_gt_orbit_mse,proj_gt_orbit_l2,orbit_abs_improvement,orbit_mse_improvement,orbit_relative_abs,best_U_index_gt,best_U_det_gt,best_U_gt
0,train,227,mp-864657,194,194,hex_trig,0.248707,0.135591,0.901967,3.116802e-08,3.297392e-15,1.406569e-07,0.248707,0.135591,1.253202e-07,2709,1,"[[0, -1, 0], [0, 1, 1], [-1, 0, 0]]"
1,train,78,mp-1221909,166,166,hex_trig,0.228013,0.120780,0.851281,1.126453e-01,3.586489e-02,4.638851e-01,0.115368,0.084915,4.940306e-01,1146,-1,"[[-1, 0, 0], [0, 1, 0], [0, -1, 1]]"
2,train,6,mp-561310,189,189,hex_trig,0.227715,0.108199,0.805725,2.030767e-08,2.369635e-15,1.192385e-07,0.227715,0.108199,8.918021e-08,2357,1,"[[0, -1, -1], [0, 0, 1], [-1, 0, 0]]"
3,train,200,mp-755568,161,161,hex_trig,0.226934,0.120201,0.849240,1.110651e-01,3.494777e-02,4.579155e-01,0.115868,0.085254,4.894167e-01,1987,1,"[[-1, 1, 0], [1, 0, 0], [0, 0, -1]]"
4,train,470,mp-1215517,166,166,hex_trig,0.226921,0.120195,0.849216,1.110465e-01,3.493704e-02,4.578452e-01,0.115874,0.085258,4.893620e-01,1188,1,"[[-1, 0, 0], [1, -1, 0], [0, -1, 1]]"


n  mean_direct_abs  mean_orbit_abs  \
split family                                              
test  hex_trig      39         0.198215        0.040268   
      orthorhombic   3         0.164763        0.163380   
      tetragonal     3         0.164025        0.151834   
      cubic         83         0.163376        0.163376   
train hex_trig      44         0.205153        0.064469   
      tetragonal     9         0.166930        0.163448   
      orthorhombic   4         0.166293        0.162901   
      cubic         71         0.163376        0.163376   
val   hex_trig      42         0.196799        0.049984   
      orthorhombic   1         0.166813        0.162631   
      tetragonal     8         0.165295        0.159656   
      cubic         77         0.163376        0.163376   

                    median_orbit_relative_abs  mean_abs_improvement  \
split family                                                          
test  hex_trig                       0.253020          1.579462e-01   
      orthorhombic                   0.999997          1.382863e-03   
      tetragonal                     0.905660          1.219141e-02   
      cubic                          0.999999          2.048461e-07   
train hex_trig                       0.398301          1.406846e-01   
      tetragonal                     0.998304          3.481577e-03   
      orthorhombic                   0.977318          3.392871e-03   
      cubic                          0.999999          2.159619e-07   
val   hex_trig                       0.272400          1.468146e-01   
      orthorhombic                   0.974930          4.182070e-03   
      tetragonal                     0.982701          5.639834e-03   
      cubic                          0.999999          2.018430e-07   

                    p95_orbit_abs  
split family                       
test  hex_trig           0.110287  
      orthorhombic       0.163569  
      tetragonal         0.161896  
      cubic              0.163376  
train hex_trig           0.112491  
      tetragonal         0.163707  
      orthorhombic       0.163356  
      cubic              0.163376  
val   hex_trig           0.111771  
      orthorhombic       0.162631  
      tetragonal         0.163904  
      cubic              0.163376

In [8]:
display(orbit_df.sort_values("proj_gt_direct_abs", ascending=False)[[
    "split", "idx", "structure_id", "sg", "family",
    "proj_gt_direct_abs", "proj_gt_orbit_abs", "orbit_abs_improvement", "orbit_relative_abs",
    "best_U_index_gt", "best_U_det_gt", "best_U_gt",
]].head(50))

display(orbit_df.sort_values("proj_gt_orbit_abs", ascending=False)[[
    "split", "idx", "structure_id", "sg", "family",
    "proj_gt_direct_abs", "proj_gt_orbit_abs", "orbit_abs_improvement", "orbit_relative_abs",
    "best_U_index_gt", "best_U_det_gt", "best_U_gt",
]].head(50))


,split,idx,structure_id,sg,family,proj_gt_direct_abs,proj_gt_orbit_abs,orbit_abs_improvement,orbit_relative_abs,best_U_index_gt,best_U_det_gt,best_U_gt
256,test,93,mp-1211616,189,hex_trig,0.259987,4.039281e-08,0.259987,1.553645e-07,2357,1,"[[0, -1, -1], [0, 0, 1], [-1, 0, 0]]"
0,train,227,mp-864657,194,hex_trig,0.248707,3.116802e-08,0.248707,1.253202e-07,2709,1,"[[0, -1, 0], [0, 1, 1], [-1, 0, 0]]"
128,val,367,mp-861867,194,hex_trig,0.235453,1.665187e-08,0.235453,7.072274e-08,2655,-1,"[[0, -1, 0], [0, 0, -1], [-1, 0, 0]]"
257,test,426,mp-768310,161,hex_trig,0.228114,1.127859e-01,0.115328,4.944279e-01,3326,-1,"[[0, 0, -1], [0, 1, 0], [-1, 1, 0]]"
1,train,78,mp-1221909,166,hex_trig,0.228013,1.126453e-01,0.115368,4.940306e-01,1146,-1,"[[-1, 0, 0], [0, 1, 0], [0, -1, 1]]"
2,train,6,mp-561310,189,hex_trig,0.227715,2.030767e-08,0.227715,8.918021e-08,2357,1,"[[0, -1, -1], [0, 0, 1], [-1, 0, 0]]"
129,val,421,mp-1225452,166,hex_trig,0.226938,1.110718e-01,0.115866,4.894368e-01,1189,-1,"[[-1, 0, 0], [1, -1, 0], [0, 0, -1]]"
3,train,200,mp-755568,161,hex_trig,0.226934,1.110651e-01,0.115868,4.894167e-01,1987,1,"[[-1, 1, 0], [1, 0, 0], [0, 0, -1]]"
4,train,470,mp-1215517,166,hex_trig,0.226921,1.110465e-01,0.115874,4.893620e-01,1188,1,"[[-1, 0, 0], [1, -1, 0], [0, -1, 1]]"
130,val,305,mp-867537,166,hex_trig,0.226860,1.109581e-01,0.115902,4.891034e-01,2780,1,"[[0, -1, 0], [1, 0, 0], [-1, 0, 1]]"


,split,idx,structure_id,sg,family,proj_gt_direct_abs,proj_gt_orbit_abs,orbit_abs_improvement,orbit_relative_abs,best_U_index_gt,best_U_det_gt,best_U_gt
162,val,391,mp-982261,140,tetragonal,0.170682,0.164158,6.523639e-03,0.961779,1092,1,"[[-1, 0, 0], [0, 0, -1], [-1, -1, 0]]"
52,train,393,mp-3248,141,tetragonal,0.163861,0.163861,2.235174e-07,0.999999,0,1,"[[1, 0, 0], [0, 1, 0], [0, 0, 1]]"
299,test,356,mp-1106355,71,orthorhombic,0.163583,0.163583,4.768372e-07,0.999997,30,1,"[[-1, -1, -1], [-1, 0, 0], [0, 0, -1]]"
56,train,325,mp-675458,122,tetragonal,0.163475,0.163475,-2.980232e-08,1.000000,3122,1,"[[0, 0, -1], [-1, -1, -1], [-1, 0, 0]]"
300,test,142,mp-1208573,71,orthorhombic,0.163445,0.163444,4.172325e-07,0.999997,2490,-1,"[[0, -1, 0], [-1, -1, -1], [-1, 0, 0]]"
172,val,511,mp-1232385,122,tetragonal,0.164764,0.163432,1.332045e-03,0.991915,934,-1,"[[-1, 0, 0], [-1, -1, -1], [0, 0, -1]]"
51,train,84,mp-675739,82,tetragonal,0.164591,0.163419,1.172155e-03,0.992878,2655,-1,"[[0, -1, 0], [0, 0, -1], [-1, 0, 0]]"
39,train,224,mp-570558,139,tetragonal,0.176062,0.163409,1.265293e-02,0.928134,2545,1,"[[0, -1, 0], [-1, 0, 0], [-1, 0, -1]]"
50,train,252,mp-1224007,42,orthorhombic,0.165210,0.163403,1.807421e-03,0.989060,2339,-1,"[[0, -1, -1], [0, 0, -1], [-1, 0, 0]]"
53,train,231,mp-37045,122,tetragonal,0.163656,0.163378,2.775639e-04,0.998304,2655,-1,"[[0, -1, 0], [0, 0, -1], [-1, 0, 0]]"


## Canonicalization Safety Check

For optional D2 final canonicalization:

```text
L_out = U* L_gen
F_out = F_gen inv(U*) mod 1
```

This should preserve Cartesian positions before wrapping. Here we test the convention using dataset fractional coordinates and decoded `L(k0)` as a stand-in for generated pairs.


In [9]:
canon_rows = []
for _, row in orbit_df.iterrows():
    sample = samples_by_key[(str(row["split"]), int(row["idx"]))]
    frac = torch.as_tensor(sample.pos, device=DEVICE).reshape(-1, 3).float()
    k0 = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
    cell = sym.v2m(k0).reshape(3, 3)
    U = torch.tensor(row["best_U_gt"], device=DEVICE, dtype=cell.dtype).reshape(3, 3)
    U_inv = torch.linalg.inv(U)
    cell_u = U @ cell
    frac_u_unwrapped = frac @ U_inv
    frac_u = frac_u_unwrapped.remainder(1.0)

    cart = frac @ cell
    cart_u_unwrapped = frac_u_unwrapped @ cell_u
    canon_rows.append({
        "split": row["split"],
        "idx": int(row["idx"]),
        "structure_id": row["structure_id"],
        "sg": int(row["sg"]),
        "family": row["family"],
        "best_U_index_gt": int(row["best_U_index_gt"]),
        "cart_equiv_max_abs_unwrapped": float((cart - cart_u_unwrapped).abs().max().item()),
        "frac_u_min": float(frac_u.min().item()),
        "frac_u_max": float(frac_u.max().item()),
    })

canon_df = pd.DataFrame(canon_rows)
display(canon_df.groupby(["split", "family"]).agg(
    n=("cart_equiv_max_abs_unwrapped", "size"),
    mean_cart_error=("cart_equiv_max_abs_unwrapped", "mean"),
    max_cart_error=("cart_equiv_max_abs_unwrapped", "max"),
).sort_values(["split", "max_cart_error"], ascending=[True, False]))

display(canon_df.sort_values("cart_equiv_max_abs_unwrapped", ascending=False).head(30))


n  mean_cart_error  max_cart_error
split family                                           
test  cubic         83     4.969447e-07    1.907349e-06
      hex_trig      39     4.187608e-07    1.907349e-06
      orthorhombic   3     5.563100e-07    9.536743e-07
      tetragonal     3     5.165736e-07    5.960464e-07
train cubic         71     4.365411e-07    9.536743e-07
      hex_trig      44     4.937703e-07    9.536743e-07
      tetragonal     9     4.635917e-07    9.536743e-07
      orthorhombic   4     4.023314e-07    4.768372e-07
val   hex_trig      42     4.512923e-07    1.430511e-06
      cubic         77     4.474219e-07    9.536743e-07
      orthorhombic   1     4.768372e-07    4.768372e-07
      tetragonal     8     4.172325e-07    4.768372e-07

,split,idx,structure_id,sg,family,best_U_index_gt,cart_equiv_max_abs_unwrapped,frac_u_min,frac_u_max
266,test,339,mp-1226990,146,hex_trig,1850,1.907349e-06,0.000728,0.991912
332,test,449,mp-1112651,225,cubic,3288,1.907349e-06,0.000000,0.778282
151,val,261,mp-14853,146,hex_trig,2588,1.430511e-06,0.000444,0.999852
0,train,227,mp-864657,194,hex_trig,2709,9.536743e-07,0.219878,0.780122
231,val,16,mp-1113055,225,cubic,3182,9.536743e-07,0.000000,0.761689
226,val,218,mp-7886,225,cubic,1800,9.536743e-07,0.000000,0.776486
223,val,293,mp-1111598,225,cubic,993,9.536743e-07,0.000000,0.781823
221,val,372,mp-25383,227,cubic,2640,9.536743e-07,-0.000000,0.999999
218,val,343,mp-1183061,225,cubic,993,9.536743e-07,0.000000,0.750000
215,val,496,mp-1187726,225,cubic,1415,9.536743e-07,0.000000,0.750000


## Direct Projection Danger Baseline

This measures the risky baseline Variant D avoids: continuously replacing `k0` with `Pi_G(k0)` while leaving fractional coordinates fixed.


In [10]:
try:
    from pymatgen.core import Lattice, Structure
    from pymatgen.analysis.structure_matcher import StructureMatcher
    PYMATGEN_MATCHER_OK = True
except Exception as exc:
    PYMATGEN_MATCHER_OK = False
    print("pymatgen/StructureMatcher unavailable:", type(exc).__name__, exc)


def structure_matcher_metrics(sample, cell_a: torch.Tensor, cell_b: torch.Tensor) -> dict[str, float | bool | None]:
    if not PYMATGEN_MATCHER_OK:
        return {"sm_fit": None, "sm_rms": None, "sm_max_dist": None}
    z = torch.as_tensor(sample.atomic_numbers).reshape(-1).detach().cpu().numpy().astype(int).tolist()
    frac = torch.as_tensor(sample.pos).reshape(-1, 3).detach().cpu().numpy()
    struct_a = Structure(Lattice(cell_a.reshape(3, 3).detach().cpu().numpy()), z, frac, coords_are_cartesian=False, to_unit_cell=True)
    struct_b = Structure(Lattice(cell_b.reshape(3, 3).detach().cpu().numpy()), z, frac, coords_are_cartesian=False, to_unit_cell=True)
    matcher = StructureMatcher(primitive_cell=False, scale=False, attempt_supercell=False)
    rms = matcher.get_rms_dist(struct_a, struct_b)
    return {
        "sm_fit": bool(matcher.fit(struct_a, struct_b)),
        "sm_rms": None if rms is None else float(rms[0]),
        "sm_max_dist": None if rms is None else float(rms[1]),
    }


In [11]:
direct_proj_rows = []
for _, row in orbit_df.iterrows():
    sample = samples_by_key[(str(row["split"]), int(row["idx"]))]
    k0 = torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float()
    sg = torch.as_tensor(sample.space_group, device=DEVICE).reshape(1).long()
    k_proj = sym.proj_k_to_spacegroup(k0, sg)
    cell0 = sym.v2m(k0).reshape(3, 3)
    cell_proj = sym.v2m(k_proj).reshape(3, 3)
    lengths0, angles0, volume0 = lengths_angles_volume(cell0)
    lengths_proj, angles_proj, volume_proj = lengths_angles_volume(cell_proj)
    direct_proj_rows.append({
        "split": row["split"],
        "idx": int(row["idx"]),
        "structure_id": row["structure_id"],
        "sg": int(row["sg"]),
        "family": row["family"],
        "proj_gt_direct_abs": float(row["proj_gt_direct_abs"]),
        "proj_gt_orbit_abs": float(row["proj_gt_orbit_abs"]),
        "length_mae_direct_projection": float((lengths0 - lengths_proj).abs().mean().item()),
        "length_max_abs_direct_projection": float((lengths0 - lengths_proj).abs().max().item()),
        "angle_mae_deg_direct_projection": float((angles0 - angles_proj).abs().mean().item()),
        "angle_max_abs_deg_direct_projection": float((angles0 - angles_proj).abs().max().item()),
        "volume_rel_error_direct_projection": float(((volume_proj - volume0) / volume0.clamp_min(1e-12)).item()),
        "fro_cell_change_direct_projection": float(torch.linalg.matrix_norm(cell0 - cell_proj).item()),
        **structure_matcher_metrics(sample, cell0, cell_proj),
    })

direct_proj_df = pd.DataFrame(direct_proj_rows)
display(direct_proj_df.groupby(["split", "family"]).agg(
    n=("proj_gt_direct_abs", "size"),
    mean_direct_abs=("proj_gt_direct_abs", "mean"),
    mean_orbit_abs=("proj_gt_orbit_abs", "mean"),
    mean_length_mae=("length_mae_direct_projection", "mean"),
    mean_angle_mae_deg=("angle_mae_deg_direct_projection", "mean"),
    mean_abs_volume_rel_error=("volume_rel_error_direct_projection", lambda s: s.abs().mean()),
    mean_fro_cell_change=("fro_cell_change_direct_projection", "mean"),
    sm_fit_rate=("sm_fit", lambda s: pd.Series(s).dropna().mean() if pd.Series(s).dropna().shape[0] else np.nan),
    mean_sm_rms=("sm_rms", "mean"),
).sort_values(["split", "mean_direct_abs"], ascending=[True, False]))


n  mean_direct_abs  mean_orbit_abs  mean_length_mae  \
split family                                                               
test  hex_trig      39         0.198215        0.040268         0.832640   
      orthorhombic   3         0.164763        0.163380         0.651905   
      tetragonal     3         0.164025        0.151834         0.603498   
      cubic         83         0.163376        0.163376         0.608163   
train hex_trig      44         0.205153        0.064469         0.565307   
      tetragonal     9         0.166930        0.163448         0.645348   
      orthorhombic   4         0.166293        0.162901         0.625197   
      cubic         71         0.163376        0.163376         0.576944   
val   hex_trig      42         0.196799        0.049984         0.710547   
      orthorhombic   1         0.166813        0.162631         0.779799   
      tetragonal     8         0.165295        0.159656         0.577328   
      cubic         77         0.163376        0.163376         0.599513   

                    mean_angle_mae_deg  mean_abs_volume_rel_error  \
split family                                                        
test  hex_trig               27.742271               2.058249e-07   
      orthorhombic           19.657072               2.133864e-07   
      tetragonal             19.545214               2.862291e-07   
      cubic                  29.746299               2.183292e-07   
train hex_trig               32.053353               1.642808e-07   
      tetragonal             19.635461               2.119174e-07   
      orthorhombic           19.895839               7.436023e-08   
      cubic                  29.406834               1.761853e-07   
val   hex_trig               28.574288               1.996792e-07   
      orthorhombic           19.934595               0.000000e+00   
      tetragonal             19.620548               1.373214e-07   
      cubic                  29.589792               2.609338e-07   

                    mean_fro_cell_change  sm_fit_rate mean_sm_rms  
split family                                                       
test  hex_trig                  4.409174          0.0         NaN  
      orthorhombic              3.611182          0.0         NaN  
      tetragonal                3.398536          0.0         NaN  
      cubic                     3.262657          0.0         NaN  
train hex_trig                  4.395431          0.0         NaN  
      tetragonal                3.560730          0.0         NaN  
      orthorhombic              3.242631          0.0         NaN  
      cubic                     3.100327          0.0         NaN  
val   hex_trig                  4.249798          0.0         NaN  
      orthorhombic              4.036710          0.0         NaN  
      tetragonal                3.188274          0.0         NaN  
      cubic                     3.218703          0.0         NaN

In [12]:
display(direct_proj_df.sort_values("fro_cell_change_direct_projection", ascending=False)[[
    "split", "idx", "structure_id", "sg", "family",
    "proj_gt_direct_abs", "proj_gt_orbit_abs",
    "length_mae_direct_projection", "angle_mae_deg_direct_projection",
    "volume_rel_error_direct_projection", "fro_cell_change_direct_projection",
    "sm_fit", "sm_rms", "sm_max_dist",
]].head(50))


,split,idx,structure_id,sg,family,proj_gt_direct_abs,proj_gt_orbit_abs,length_mae_direct_projection,angle_mae_deg_direct_projection,volume_rel_error_direct_projection,fro_cell_change_direct_projection,sm_fit,sm_rms,sm_max_dist
256,test,93,mp-1211616,189,hex_trig,0.259987,4.039281e-08,3.736814,20.000000,-1.439270e-07,9.415921,False,None,None
0,train,227,mp-864657,194,hex_trig,0.248707,3.116802e-08,2.864960,19.999994,-6.408398e-08,7.347680,False,None,None
128,val,367,mp-861867,194,hex_trig,0.235453,1.665187e-08,2.669817,20.000008,1.791035e-07,7.038651,False,None,None
133,val,396,mp-1247221,166,hex_trig,0.223941,1.114314e-01,0.451307,39.204205,1.919624e-07,6.249157,False,None,None
266,test,339,mp-1226990,146,hex_trig,0.218221,9.852844e-02,0.414169,38.132801,-9.045627e-08,6.164264,False,None,None
136,val,107,mp-1220114,166,hex_trig,0.220508,1.113636e-01,0.411544,38.572147,-1.987636e-07,6.082651,False,None,None
5,train,281,mp-1247173,166,hex_trig,0.226680,1.125161e-01,0.456927,39.730503,-2.239623e-07,6.017668,False,None,None
1,train,78,mp-1221909,166,hex_trig,0.228013,1.126453e-01,0.465448,39.981289,-1.158493e-07,5.987104,False,None,None
9,train,170,mp-1540553,166,hex_trig,0.225946,1.123822e-01,0.448017,39.594055,-3.393501e-07,5.977125,False,None,None
258,test,229,mp-1226282,160,hex_trig,0.226313,1.101589e-01,0.449027,39.662548,-1.168017e-07,5.916955,False,None,None


## Decision Checklist

Variant D is promising if:

- `mean_orbit_abs` is much smaller than `mean_direct_abs`.
- `orbit_relative_abs` is often far below `1.0`.
- `cart_equiv_max_abs_unwrapped` is near numerical zero for D2 canonicalization.
- Direct projection shows meaningful geometry or StructureMatcher changes.

If this passes, the next implementation should be:

```text
D1 = raw k target + small orbit SG auxiliary + normal sampling
D2 = D1 + optional final U* canonicalization
```

Start small: `lambda_orbit = 0.01`, then try `0.05` and `0.1`.
